Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [16]:
NO_RECORDS = 150

In [17]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

In [18]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [19]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [20]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [21]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1931', 'title': 'Data publication: Path integral Monte Carlo results for electrons in 2D and 3D quantum dots', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:1931</identifier>\n    <datestamp>2022-11-08T10:55:12Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n         

In [22]:
record_titles

['Data publication: Path integral Monte Carlo results for electrons in 2D and 3D quantum dots',
 'Data publication: η-ensemble path integral Monte Carlo approach to the free energy of the warm dense electron gas and the uniform electron liquid',
 'Data publication: Laboratory investigation of tomography-controlled continuous steel casting',
 'Data: Tuning transport properties of conducting ceramic through point defects',
 'Data for: "A metabolic switch regulates the transition between growth and diapause in C. elegans"',
 'Data and code for: Mitigating pseudoreplication and bias in resource selection functions with autocorrelation-informed weighting',
 'Data publication: Nonreciprocal spin-wave dispersion in magnetic bilayers',
 'APR files',
 'Raw data, magnetization measurements for paper Room temperature ferromagnetism in Sb doped ZnO',
 'Data publication: Unraveling electronic correlations in warm dense quantum plasmas',
 'Data publication: Scalable Machine Learning Model for Energy

In [23]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:1463</identifier>\n    <datestamp>2022-03-07T07:36:03Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-ecfunded</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.1463</identifier>\n          <creators>\n            <creator>\n              <creator

In [24]:
get_doi(record_raw_metadata[2])

'10.14278/rodare.1463'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [25]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [26]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

[]


In [27]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [28]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['a64cfb20-e461-4ec0-a069-a00c1fde6c64::1931', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3765', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1463', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3947', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::760', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1862', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3648', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1135', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1001', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3674', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4170', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2559', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2279', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::281', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3323', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1369', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1836', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4596', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3866', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2778', 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3593', 'a64cfb20-e461-4ec0-a069-a00c1fde6c

In [29]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [30]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Xu, Chi', 'name': 'Chi', 'surname': 'Xu', 'rank': 1, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-1299-7667'}, 'provenance': None}}, {'fullName': 'Zhou, Shengqiang', 'name': 'Shengqiang', 'surname': 'Zhou', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-4885-799x'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': None, 'mainTitle': 'Raw data, magnetization measurements for paper Room temperature ferromagnetism in Sb doped ZnO', 'subTitle': None, 'descriptions': ['Raw data, magnetization measurements for samples involved in paper Room temperature ferromagnetism in Sb doped ZnO'], 'publicationDate': '2021-01-01', 'publisher': 'Rodare', 'embargoEndDate': None, 'sources': None, 'formats': None, 'contributors': None, 'coverages': None, 'bestAccessRight': {'code': 'c_abf2', 'label': 'OPEN', 'scheme': 

In [31]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.14278/rodare.357', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.1192', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.358', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.1098', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.1958', 'subjects': ['13. Climate action']},
 {'doi': '10.14278/rodare.1959', 'subjects': ['13. Climate action']},
 {'doi': '10.14278/rodare.771',
  'subjects': ['13. Climate action', '6. Clean water']},
 {'doi': '10.14278/rodare.772',
  'subjects': ['13. Climate action', '6. Clean water']},
 {'doi': '10.14278/rodare.502', 'subjects': ['7. Clean energy']},
 {'doi': '10.14278/rodare.503', 'subjects': ['7. Clean energy']},
 {'doi': '10.14278/rodare.357', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.1192', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.358', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.1098', 'subjects': ['3. Good health']},
 {'doi': '10.14278/rodare.

In [32]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1931',
  'doi': '10.14278/rodare.1931'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3765',
  'doi': '10.14278/rodare.3765'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1463',
  'doi': '10.14278/rodare.1463'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3947',
  'doi': '10.14278/rodare.3947'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::760',
  'doi': '10.14278/rodare.760'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1862',
  'doi': '10.14278/rodare.1862'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3648',
  'doi': '10.14278/rodare.3648'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1135',
  'doi': '10.14278/rodare.1135'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1001',
  'doi': '10.14278/rodare.1001'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3674',
  'doi': '10.14278/rodare.3674'},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4170',
  'doi': '10.14278/rodare.4170'},
 {'id': 'a64cfb20-e461-4ec0-a069-a

In [33]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1931',
  'doi': '10.14278/rodare.1931',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3765',
  'doi': '10.14278/rodare.3765',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1463',
  'doi': '10.14278/rodare.1463',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3947',
  'doi': '10.14278/rodare.3947',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::760',
  'doi': '10.14278/rodare.760',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1862',
  'doi': '10.14278/rodare.1862',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3648',
  'doi': '10.14278/rodare.3648',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1135',
  'doi': '10.14278/rodare.1135',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1001',
  'd

In [34]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1931',
  'doi': '10.14278/rodare.1931',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3765',
  'doi': '10.14278/rodare.3765',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1463',
  'doi': '10.14278/rodare.1463',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3947',
  'doi': '10.14278/rodare.3947',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::760',
  'doi': '10.14278/rodare.760',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1862',
  'doi': '10.14278/rodare.1862',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3648',
  'doi': '10.14278/rodare.3648',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1135',
  'doi': '10.14278/rodare.1135',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1001',
  'd

In [35]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1931',
  'doi': '10.14278/rodare.1931',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3765',
  'doi': '10.14278/rodare.3765',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1463',
  'doi': '10.14278/rodare.1463',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3947',
  'doi': '10.14278/rodare.3947',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::760',
  'doi': '10.14278/rodare.760',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1862',
  'doi': '10.14278/rodare.1862',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3648',
  'doi': '10.14278/rodare.3648',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1135',
  'doi': '10.14278/rodare.1135',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1001',
  'd

In [36]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

150
150


In [37]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [38]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [39]:
first_enriched

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1931',
  'doi': '10.14278/rodare.1931',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3765',
  'doi': '10.14278/rodare.3765',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1463',
  'doi': '10.14278/rodare.1463',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3947',
  'doi': '10.14278/rodare.3947',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::760',
  'doi': '10.14278/rodare.760',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1862',
  'doi': '10.14278/rodare.1862',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3648',
  'doi': '10.14278/rodare.3648',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1135',
  'doi': '10.14278/rodare.1135',
  'enriched_subjects': set()},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1001',
  'd

In [40]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:1931</identifier>\n    <datestamp>2022-11-08T10:55:12Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.1931</identifier>\n          <creators>\n            <creator>\n              <creatorName>Dornheim, Tobias</creatorName>\n

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [41]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [42]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [43]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [44]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

31


[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1001',
  'doi': '10.14278/rodare.1001',
  'enriched_subjects': {'Doping',
   'Ferromagnetism',
   'Magnetic Field Sensors Techniques',
   'Magnetization',
   'Materials science'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::4170',
  'doi': '10.14278/rodare.4170',
  'enriched_subjects': {'Artificial neural network'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::281',
  'doi': '10.14278/rodare.281',
  'enriched_subjects': {'Bridging (networking)',
   'Copper Interconnects and Reliability',
   'Gas Sensing Nanomaterials and Sensors',
   'Silicone and Siloxane Chemistry',
   'Terminal (telecommunication)'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::715',
  'doi': '10.14278/rodare.715',
  'enriched_subjects': {'Metal Extraction and Bioleaching', 'Skarn'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::1053',
  'doi': '10.14278/rodare.1053',
  'enriched_subjects': {'Graphene research and applications',
   'Ion',
   'Ion-surface int

In [45]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

In [46]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [47]:
def merge_subjects_from_new_method(base_list, supplement_list):
    """
    Extend enriched_subjects from supplement_list to matching objects in base_list. Matches by id.
    Modifies in-place.
    """
    supplement_index = {item["id"]: item["enriched_subjects"] for item in supplement_list}

    for item in base_list:
        additional_subjects = supplement_index.get(item["id"])

        if additional_subjects:
            item["enriched_subjects"] = item["enriched_subjects"] | additional_subjects

    return base_list

In [48]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword